In [ ]:
import pandas as pd
import numpy as np
from numpy import mean
from numpy import std
import os
import math
import csv
import shutil, sys
import torch
from torchvision import transforms

from PIL import Image

from google.colab import drive
drive.mount('/content/drive')
jj = '/content/drive/My Drive/datasets/'

In [ ]:
df = pd.read_csv(jj + 'LLM_Zh_Prompt.csv') 
df

In [ ]:
import os, re, json, math, pandas as pd, torch
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "hfl/chinese-llama-2-7b"

bnb_config = dict(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    **bnb_config
)
model.eval()

In [ ]:
N_SAMPLES = 10
MAX_NEW_TOKENS = 16
TEMPERATURE = 1.0

anchor_col = "llama2Chinese_utt_can"
insert_at = df.columns.get_loc(anchor_col) + 1 if anchor_col in df.columns else len(df.columns)

col_answer   = "llama2Chinese_utt_can_bin_answer"       
col_conf     = "llama2Chinese_utt_can_bin_conf"         
col_counts   = "llama2Chinese_utt_can_bin_counts"       
col_mapping  = "llama2Chinese_utt_can_bin_mapping"      

for offset, (cn, fill) in enumerate([
    (col_answer,  None),
    (col_conf,    None),
    (col_counts,  None),
    (col_mapping, None),
    (col_prompt,  None),
]):
    if cn not in df.columns:
        df.insert(insert_at + offset, cn, [None]*len(df))

for i, row in df.iterrows():

    if random.random() < 0.5:
        label_map = {"A": "可以", "B": "不能"}
    else:
        label_map = {"A": "不能", "B": "可以"}

    context   = str(row["context"])
    utterance = str(row["utterance"])

    prompt = build_prompt_binary(context, utterance, label_map)

    raw_samples = sample_predictions(
        prompt,
        n=N_SAMPLES,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE
    )
    extracted = [extract_choice_ab(s) for s in raw_samples]
    vote, conf, counts = majority_vote_ab(extracted)

    # Update DataFrame row
    df.at[i, col_answer]  = vote
    df.at[i, col_conf]    = conf
    df.at[i, col_counts]  = json.dumps(counts, ensure_ascii=False)
    df.at[i, col_mapping] = f"A={label_map['A']},B={label_map['B']}"
    df.at[i, col_prompt]  = prompt  

    df.to_csv(jj + 'LLM_Zh_Prompt.csv', index=False, encoding="utf-8")

    semantic = None
    if vote in ("A","B"):
        semantic = label_map[vote]
    print(
        f"[Row {i+1}/{len(df)}] vote={vote} ({semantic}), "
        f"conf={conf:.2f}, counts={counts}, mapping=A:{label_map['A']} B:{label_map['B']}"
    )

print("Binary task completed. All rows processed and saved.")

In [ ]:
df[['llama2Chinese_utt_can_bin_answer', 'llama2Chinese_utt_can_bin_conf', 'llama2Chinese_utt_can_bin_counts','llama2Chinese_utt_can_bin_mapping']].head(1)

,llama2Chinese_utt_can_bin_answer,llama2Chinese_utt_can_bin_conf,llama2Chinese_utt_can_bin_counts,llama2Chinese_utt_can_bin_mapping
0,A,1.0,"{""A"": 7}","A=不能,B=可以"


In [ ]:
N_SAMPLES = 10
MAX_NEW_TOKENS = 16
TEMPERATURE = 1.0

anchor_col = "llama2Chinese_no_qud_alt"
insert_at = df.columns.get_loc(anchor_col) + 1 if anchor_col in df.columns else len(df.columns)

df.insert(insert_at,     "llama2Chinese_no_qud_alt_answer", [None]*len(df))
df.insert(insert_at + 1, "llama2Chinese_no_qud_alt_conf",   [None]*len(df))
df.insert(insert_at + 2, "llama2Chinese_no_qud_alt_counts", [None]*len(df))

for i, row in df.iterrows():
    options = {
        "A": row["utterance"],
        "B": row["utterance_hen"],
        "C": row["utterance_bijiao"],
        "D": row["utterance_bijiao_neg_pos"],
        "E": row["utterance_hen_neg_comp"],
    }
    prompt = build_prompt(row["context"], options)

    raw_samples = sample_predictions(
        prompt,
        n=N_SAMPLES,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE
    )
    extracted = [extract_choice(s) for s in raw_samples]
    vote, conf, counts = majority_vote(extracted)

    df.at[i, "llama2Chinese_no_qud_alt_answer"] = vote
    df.at[i, "llama2Chinese_no_qud_alt_conf"]   = conf
    df.at[i, "llama2Chinese_no_qud_alt_counts"] = json.dumps(counts, ensure_ascii=False)

    df.to_csv(jj + 'LLM_Zh_Prompt.csv', index=False, encoding="utf-8")

    print(f"[Row {i+1}/{len(df)} finished] Answer={vote}, Conf={conf:.2f}, Counts={counts}")

print("All rows processed and saved successfully!")

In [ ]:
df[['llama2Chinese_no_qud_alt_answer', 'llama2Chinese_no_qud_alt_conf', 'llama2Chinese_no_qud_alt_counts']].head(1)

,llama2Chinese_no_qud_alt_answer,llama2Chinese_no_qud_alt_conf,llama2Chinese_no_qud_alt_counts
0,A,0.6,"{""A"": 3, ""E"": 1, ""B"": 1}"
